# Regio-AI: Stage 2 — Prithvi Water Detection & Strandskydd Violation Mapping

**Project:** Fine-tuning IBM/NASA Prithvi for Strandskydd Violation Detection  
**Platform:** OpenShift AI — PyTorch notebook (CPU)  
**Model:** `ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL-Sen1Floods11`  
**Stage:** 2 of N — Foundation Model Inference & Violation Detection  

> **Prerequisite:** Run the Stage 1 notebook first to understand the full data pipeline.  
> This notebook focuses Prithvi inference on a 224×224 pixel patch centred on the Point of Interest.

---

## What this notebook does

1. Downloads the same Sentinel-2 before/after scenes as Stage 1
2. Loads **IBM/NASA Prithvi-EO-2.0-300M** (~1.2 GB, cached after first run)
3. Runs Prithvi on a patch centred on the POI to detect water bodies  
4. Derives the strandskydd 100 m protection zone from the Prithvi water mask  
5. Uses NDBI change detection to identify new built-up surfaces between scenes  
6. Intersects the two to flag potential strandskydd violations

---

## 1. Environment Setup

`terratorch` is IBM's official framework for Prithvi EO models.  
It includes PyTorch Lightning and torchgeo — first install takes several minutes.

In [ ]:
%pip install --quiet terratorch timm einops huggingface-hub
%pip install --quiet planetary-computer pystac-client stackstac rasterio geopandas shapely folium pyproj
print("Installation complete.")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import geopandas as gpd
import folium
from folium import plugins
from shapely.geometry import shape
import rasterio
import rasterio.features
from rasterio.features import shapes as rasterio_shapes
from rasterio.enums import Resampling
from affine import Affine
from pyproj import Transformer
import torch
import pystac_client
import planetary_computer
import stackstac

print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {'GPU' if torch.cuda.is_available() else 'CPU'}")
print("Imports OK.")

In [ ]:
# --- Area of Interest ---
AOI_NAME  = "Orust, Bohuslän, West Coast of Sweden"
AOI_BBOX  = {"west": 11.65, "south": 58.18, "east": 11.91, "north": 58.35}
BBOX_LIST = [AOI_BBOX["west"], AOI_BBOX["south"], AOI_BBOX["east"], AOI_BBOX["north"]]
AOI_CENTER = [(AOI_BBOX["south"] + AOI_BBOX["north"]) / 2,
              (AOI_BBOX["west"]  + AOI_BBOX["east"])  / 2]

# --- Point of Interest ---
POI_LAT   = 58.26599
POI_LON   = 11.77902
POI_LABEL = "Summer house location — suspected new structure post-2019"

# --- Strandskydd ---
STRANDSKYDD_DISTANCE_M = 100

# --- Sentinel-2 ---
PRITHVI_BANDS   = ["B02", "B03", "B04", "B8A", "B11", "B12"]
MAX_CLOUD_COVER = 20
TEMPORAL_AFTER  = ["2022-06-01", "2023-09-30"]
TEMPORAL_BEFORE = ["2017-01-01", "2018-12-31"]
INVALID_SCL     = {0, 1, 3, 8, 9, 10, 11}

# --- Prithvi ---
MODEL_ID    = "ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL-Sen1Floods11"
PATCH_SIZE  = 224   # pixels (2.24 km × 2.24 km at 10 m/px)

# Normalisation stats from Sen1Floods11 model card (applied to raw DN 0-10000)
# Band order: Blue(B02) Green(B03) Red(B04) NarrowNIR(B8A) SWIR1(B11) SWIR2(B12)
PRITHVI_MEANS = np.array([1087.0, 1342.0, 1433.0, 2734.0, 1958.0, 1363.0])
PRITHVI_STDS  = np.array([2248.0, 2179.0, 2178.0, 1850.0, 1242.0, 1049.0])

print(f"AOI    : {AOI_NAME}")
print(f"POI    : {POI_LAT}°N, {POI_LON}°E")
print(f"Model  : {MODEL_ID}")
print(f"Device : {'GPU' if torch.cuda.is_available() else 'CPU'}")

## 2. Sentinel-2 Data

Same before/after scenes as Stage 1 — loads quickly from Planetary Computer (COGs).

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

def best_scene(temporal_extent, label):
    items = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=BBOX_LIST,
        datetime=f"{temporal_extent[0]}/{temporal_extent[1]}",
        query={"eo:cloud_cover": {"lt": MAX_CLOUD_COVER}},
    ).item_collection()
    best = min(items, key=lambda x: x.properties["eo:cloud_cover"])
    print(f"{label}: {best.datetime.strftime('%Y-%m-%d')}  "
          f"cloud={best.properties['eo:cloud_cover']:.1f}%  "
          f"tile={best.properties.get('s2:mgrs_tile','—')}")
    return best

best_after  = best_scene(TEMPORAL_AFTER,  "'After' ")
best_before = best_scene(TEMPORAL_BEFORE, "'Before'")

In [ ]:
def make_stack(item):
    return stackstac.stack([item], assets=PRITHVI_BANDS, bounds_latlon=BBOX_LIST,
                           resolution=10, dtype="float64", fill_value=np.nan,
                           rescale=False, epsg=32633)

def make_scl(item):
    return stackstac.stack([item], assets=["SCL"], bounds_latlon=BBOX_LIST,
                           resolution=10, dtype="float64", fill_value=0,
                           rescale=False, epsg=32633, resampling=Resampling.nearest)

print("Downloading scenes and SCL masks...")
data_after  = make_stack(best_after).squeeze("time").compute()
data_before = make_stack(best_before).squeeze("time").compute()
scl_after   = make_scl(best_after).squeeze("time").isel(band=0).compute().values
scl_before  = make_scl(best_before).squeeze("time").isel(band=0).compute().values

date_after  = best_after.datetime.strftime("%Y-%m-%d")
date_before = best_before.datetime.strftime("%Y-%m-%d")

valid_after  = ~np.isin(scl_after,  list(INVALID_SCL))
valid_before = ~np.isin(scl_before, list(INVALID_SCL))
print(f"After  ({date_after})  : {data_after.nbytes/1e6:.0f} MB  |  {valid_after.mean()*100:.1f}% valid pixels")
print(f"Before ({date_before}) : {data_before.nbytes/1e6:.0f} MB  |  {valid_before.mean()*100:.1f}% valid pixels")

## 3. Load IBM/NASA Prithvi-EO-2.0-300M

`Prithvi-EO-2.0-300M-TL-Sen1Floods11` is a 300M-parameter Vision Transformer  
fine-tuned for water/flood segmentation on the Sen1Floods11 dataset  
(446 labelled scenes, 14 biomes, 6 continents, 88%+ mIoU).

- **First download:** ~1.2 GB — cached by HuggingFace after first run  
- **Inference time on CPU:** ~1–3 minutes per 224×224 patch

In [ ]:
from huggingface_hub import snapshot_download

print(f"Downloading {MODEL_ID}...")
print("(~1.2 GB — subsequent runs use cache)\n")
t0 = time.time()
model_dir = snapshot_download(MODEL_ID)
print(f"Ready in {time.time()-t0:.0f}s  →  {model_dir}\n")

# Locate the checkpoint file
ckpt_path = None
for root, _, files in os.walk(model_dir):
    for f in files:
        if f.endswith(('.ckpt', '.pt', '.pth')):
            ckpt_path = os.path.join(root, f)
            size_mb   = os.path.getsize(ckpt_path) / 1e6
            print(f"Checkpoint : {f}  ({size_mb:.0f} MB)")
            break
    if ckpt_path:
        break

if ckpt_path is None:
    print("No checkpoint found. Files in model dir:")
    for root, _, files in os.walk(model_dir):
        for f in files:
            print(f"  {os.path.relpath(os.path.join(root, f), model_dir)}")
    raise FileNotFoundError("Could not locate a .ckpt / .pt / .pth file.")

In [ ]:
from terratorch.tasks import SemanticSegmentationTask

print("Loading model into CPU memory...")
t0 = time.time()
task = SemanticSegmentationTask.load_from_checkpoint(
    ckpt_path,
    map_location="cpu",
    strict=False,
)
model = task.model.eval()
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Loaded in {time.time()-t0:.1f}s  |  {n_params:.0f}M parameters  |  device: CPU")

## 4. Extract POI-Centred Patch

We extract a **224×224 pixel tile centred on the Point of Interest**.  
At 10 m/pixel this covers **2.24 km × 2.24 km** — enough to capture the  
summer house and the surrounding water that defines the strandskydd zone.

Processing the full AOI on CPU would take hours; a single patch takes minutes.

In [ ]:
# Project POI (WGS84) → UTM 33N to match raster coordinates
to_utm = Transformer.from_crs("EPSG:4326", "EPSG:32633", always_xy=True)
poi_x_utm, poi_y_utm = to_utm.transform(POI_LON, POI_LAT)

x_coords = data_after.x.values
y_coords = data_after.y.values
poi_col   = int(np.argmin(np.abs(x_coords - poi_x_utm)))
poi_row   = int(np.argmin(np.abs(y_coords - poi_y_utm)))

half      = PATCH_SIZE // 2
row_start = max(0, poi_row - half)
col_start = max(0, poi_col - half)
row_end   = min(row_start + PATCH_SIZE, data_after.shape[-2])
col_end   = min(col_start + PATCH_SIZE, data_after.shape[-1])

def extract_patch(data):
    """Extract and pad a (6, PATCH_SIZE, PATCH_SIZE) float32 array."""
    bands = [data.sel(band=b).values[row_start:row_end, col_start:col_end]
             for b in PRITHVI_BANDS]
    patch = np.stack(bands, axis=0).astype(np.float32)
    ph = PATCH_SIZE - patch.shape[1]
    pw = PATCH_SIZE - patch.shape[2]
    if ph > 0 or pw > 0:
        patch = np.pad(patch, ((0,0),(0,ph),(0,pw)), mode='reflect')
    return patch

patch_after  = extract_patch(data_after)
patch_before = extract_patch(data_before)

# Reconstruct the affine transform for the patch origin (in UTM)
dx = float(x_coords[1] - x_coords[0])
dy = float(y_coords[1] - y_coords[0])   # negative (top → bottom)
patch_transform = Affine(
    dx,  0, float(x_coords[col_start]) - dx / 2,
     0, dy, float(y_coords[row_start]) - dy / 2
)
utm_crs = "EPSG:32633"

print(f"POI (UTM 33N)    : {poi_x_utm:.0f} E, {poi_y_utm:.0f} N")
print(f"POI pixel        : row={poi_row}, col={poi_col}")
print(f"Patch extent     : rows {row_start}:{row_end}, cols {col_start}:{col_end}")
print(f"Patch shape      : {patch_after.shape}  (bands × H × W)")
print(f"Spatial coverage : {PATCH_SIZE * 10 / 1000:.2f} km × {PATCH_SIZE * 10 / 1000:.2f} km")

In [ ]:
def normalize_patch(patch_dn):
    """Normalise DN (0-10000) → Prithvi input using training statistics."""
    means = PRITHVI_MEANS[:, None, None].astype(np.float32)
    stds  = PRITHVI_STDS[:, None, None].astype(np.float32)
    return (patch_dn - means) / stds

patch_after_norm  = normalize_patch(patch_after)
patch_before_norm = normalize_patch(patch_before)

def stretch(arr, p=2):
    valid = arr[np.isfinite(arr) & (arr > 0)]
    if len(valid) == 0:
        return np.zeros_like(arr)
    lo, hi = np.percentile(valid, [p, 100 - p])
    return np.clip((arr - lo) / (hi - lo + 1e-10), 0, 1)

# Band indices: 0=B02(Blue) 1=B03(Green) 2=B04(Red)
rgb_after  = np.dstack([stretch(patch_after[2]),  stretch(patch_after[1]),  stretch(patch_after[0])])
rgb_before = np.dstack([stretch(patch_before[2]), stretch(patch_before[1]), stretch(patch_before[0])])

cx = cy = PATCH_SIZE // 2  # POI is at patch centre
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, rgb, label, col in [
    (axes[0], rgb_before, f"BEFORE — {date_before}", "steelblue"),
    (axes[1], rgb_after,  f"AFTER  — {date_after}",  "firebrick"),
]:
    ax.imshow(rgb)
    ax.plot(cx, cy, 'r*', markersize=20, label="POI")
    ax.set_title(label, fontsize=13, color=col)
    ax.axis("off")
    ax.legend(loc="upper right", fontsize=11)
plt.suptitle(f"POI Patch (224×224 px  |  10 m/px  |  2.24 km²)\n{AOI_NAME}", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
print("Patches normalised and ready for Prithvi inference.")

## 5. Prithvi Inference — Water Detection

Each forward pass takes **1–3 minutes on CPU** for a 224×224 patch.

**Output classes:**
- Class 0 — No water / dry land  
- Class 1 — Water / flood

In [ ]:
def run_prithvi(patch_norm, label=""):
    """Run Prithvi on a normalised (6, 224, 224) float32 patch. Returns (preds, probs)."""
    tensor = torch.from_numpy(patch_norm).float().unsqueeze(0)  # (1, 6, 224, 224)
    print(f"Running Prithvi on {label} patch …", end=" ", flush=True)
    t0 = time.time()
    with torch.no_grad():
        outputs = model(tensor)
    elapsed = time.time() - t0

    # Handle different output types: plain tensor, HF ModelOutput, list/tuple
    if isinstance(outputs, torch.Tensor):
        logits = outputs
    elif hasattr(outputs, 'logits'):
        logits = outputs.logits
    elif hasattr(outputs, 'output'):
        logits = outputs.output
    elif isinstance(outputs, (list, tuple)):
        logits = outputs[0]
    else:
        # ModelOutput is dict-like — grab first value via .values()
        try:
            logits = next(iter(outputs.values()))
        except (AttributeError, StopIteration):
            raise TypeError(
                f"Cannot extract logits from {type(outputs).__name__}. "
                f"Available attributes: {[k for k in dir(outputs) if not k.startswith('_')]}"
            )

    probs = torch.softmax(logits, dim=1).squeeze(0).numpy()  # (2, H, W)
    preds = logits.argmax(dim=1).squeeze(0).numpy()           # (H, W)
    water_px = (preds == 1).sum()
    print(f"done in {elapsed:.1f}s  |  water: {water_px:,} / {preds.size:,} px ({water_px/preds.size*100:.1f}%)")
    return preds, probs

preds_after,  probs_after  = run_prithvi(patch_after_norm,  f"after  ({date_after})")
preds_before, probs_before = run_prithvi(patch_before_norm, f"before ({date_before})")

In [ ]:
water_after  = preds_after  == 1
water_before = preds_before == 1

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for row, (rgb, water, probs, label, col) in enumerate([
    (rgb_before, water_before, probs_before, f"BEFORE — {date_before}", "steelblue"),
    (rgb_after,  water_after,  probs_after,  f"AFTER  — {date_after}",  "firebrick"),
]):
    axes[row, 0].imshow(rgb)
    axes[row, 0].plot(cx, cy, 'r*', markersize=15)
    axes[row, 0].set_title(f"RGB — {label}", fontsize=11, color=col)
    axes[row, 0].axis("off")

    axes[row, 1].imshow(water, cmap="Blues")
    axes[row, 1].plot(cx, cy, 'r*', markersize=15)
    axes[row, 1].set_title(f"Prithvi water mask\n{water.sum():,} px ({water.mean()*100:.1f}%)", fontsize=11)
    axes[row, 1].axis("off")

    im = axes[row, 2].imshow(probs[1], cmap="Blues", vmin=0, vmax=1)
    axes[row, 2].plot(cx, cy, 'r*', markersize=15)
    axes[row, 2].set_title("Water probability", fontsize=11)
    axes[row, 2].axis("off")
    plt.colorbar(im, ax=axes[row, 2], fraction=0.046, pad=0.04)

plt.suptitle(f"Prithvi Water Detection — {AOI_NAME}", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 6. Strandskydd 100 m Buffer from Prithvi Water Mask

Vectorise the Prithvi water mask for the **after** scene and apply a 100 m buffer  
in UTM coordinates (metres). This replaces the NDWI-derived buffer from Stage 1  
with a foundation-model-derived one.

In [ ]:
water_geoms = [
    shape(geom)
    for geom, val in rasterio_shapes(water_after.astype(np.uint8), transform=patch_transform)
    if val == 1
]
print(f"Prithvi water polygons : {len(water_geoms):,}")

strandskydd_patch_gdf = None
water_patch_wgs84     = None
strandskydd_mask      = np.zeros((PATCH_SIZE, PATCH_SIZE), dtype=bool)

if water_geoms:
    water_gdf      = gpd.GeoDataFrame(geometry=water_geoms, crs=utm_crs)
    water_dissolved = water_gdf.dissolve()

    water_patch_wgs84 = (
        water_dissolved.simplify(20).to_frame("geometry")
        .set_crs(utm_crs).to_crs("EPSG:4326")
    )
    strandskydd_patch_gdf = (
        gpd.GeoDataFrame(geometry=water_dissolved.buffer(STRANDSKYDD_DISTANCE_M), crs=utm_crs)
        .to_crs("EPSG:4326")
    )

    # Rasterise the buffer onto the patch grid for pixel-level intersection
    strandskydd_utm = strandskydd_patch_gdf.to_crs(utm_crs)
    strandskydd_raster = rasterio.features.rasterize(
        [(geom, 1) for geom in strandskydd_utm.geometry],
        out_shape=(PATCH_SIZE, PATCH_SIZE),
        transform=patch_transform,
        fill=0, dtype=np.uint8
    )
    strandskydd_mask = strandskydd_raster == 1

    area_km2 = strandskydd_patch_gdf.to_crs(utm_crs).area.sum() / 1e6
    print(f"Strandskydd buffer     : {STRANDSKYDD_DISTANCE_M} m  ({area_km2:.3f} km²)")
    print(f"Protected pixels       : {strandskydd_mask.sum():,} / {PATCH_SIZE**2:,} ({strandskydd_mask.mean()*100:.1f}%)")
else:
    print("No water detected — check model output and normalisation stats.")

## 7. NDBI Change Detection — New Built-Up Surfaces

We compute the Normalised Difference Built-up Index (NDBI) on the POI patch for  
both scenes and take the difference. A significant positive change flags pixels  
that became more built-up between the before and after dates.

$$\text{NDBI} = \frac{\text{SWIR}_1 - \text{NIR}}{\text{SWIR}_1 + \text{NIR}}$$

In [ ]:
def compute_ndbi(swir1, nir):
    s, n = swir1.astype(float), nir.astype(float)
    d = s + n; d[d == 0] = np.nan
    return (s - n) / d

# Band indices in patch: 3=B8A(NIR), 4=B11(SWIR1)
ndbi_after  = compute_ndbi(patch_after[4],  patch_after[3])
ndbi_before = compute_ndbi(patch_before[4], patch_before[3])

# Apply SCL cloud masks to the patch region
scl_p_after  = scl_after[row_start:row_end,  col_start:col_end]
scl_p_before = scl_before[row_start:row_end, col_start:col_end]
# Pad SCL patches if needed
for arr, target in [(scl_p_after, 'a'), (scl_p_before, 'b')]:
    pass  # shapes already match since patch was extracted identically
ndbi_after[ ~(~np.isin(scl_p_after,  list(INVALID_SCL)))] = np.nan
ndbi_before[~(~np.isin(scl_p_before, list(INVALID_SCL)))] = np.nan

ndbi_change  = ndbi_after - ndbi_before
NDBI_THRESHOLD = 0.05
new_buildup  = (ndbi_change > NDBI_THRESHOLD) & np.isfinite(ndbi_change)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
im0 = axes[0].imshow(ndbi_before, cmap="RdYlGn_r", vmin=-0.4, vmax=0.4)
axes[0].set_title(f"NDBI — BEFORE ({date_before})"); axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046)
axes[0].plot(cx, cy, 'r*', markersize=15)

im1 = axes[1].imshow(ndbi_after, cmap="RdYlGn_r", vmin=-0.4, vmax=0.4)
axes[1].set_title(f"NDBI — AFTER ({date_after})"); axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046)
axes[1].plot(cx, cy, 'r*', markersize=15)

im2 = axes[2].imshow(ndbi_change, cmap="seismic", vmin=-0.3, vmax=0.3)
axes[2].set_title(f"NDBI Change (After − Before)\nred = new built-up  |  threshold={NDBI_THRESHOLD}")
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.046)
axes[2].plot(cx, cy, 'r*', markersize=15)

plt.suptitle(f"NDBI Change Detection — {date_before} → {date_after}", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()
print(f"New built-up pixels (NDBI Δ > {NDBI_THRESHOLD}): {new_buildup.sum():,}  "
      f"(~{new_buildup.sum() * 100:.0f} m²)")

## 8. Strandskydd Violation Detection

A **potential violation** is any pixel that is:

- Newly built-up (NDBI increased significantly since the before scene), **AND**
- Inside the Prithvi-derived 100 m strandskydd protection zone

These pixels are flagged for human review — not definitive violations, but locations  
warranting investigation by a *handläggare* (case officer).

In [ ]:
violation_mask = new_buildup & strandskydd_mask

print(f"Strandskydd zone   : {strandskydd_mask.sum():,} px")
print(f"New built-up       : {new_buildup.sum():,} px")
print(f"POTENTIAL VIOLATIONS: {violation_mask.sum():,} px  (~{violation_mask.sum() * 100:.0f} m²)")
if violation_mask.any():
    print("\n>>> Potential strandskydd violation detected near the POI.")
else:
    print("\n>>> No violation signal detected in this patch / date combination.")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(rgb_after)
axes[0].plot(cx, cy, 'r*', markersize=15)
axes[0].set_title(f"RGB — {date_after}"); axes[0].axis("off")

axes[1].imshow(rgb_after)
if strandskydd_mask.any():
    axes[1].imshow(np.ma.masked_where(~strandskydd_mask, strandskydd_mask.astype(float)),
                   cmap="Reds", alpha=0.35)
if new_buildup.any():
    axes[1].imshow(np.ma.masked_where(~new_buildup, new_buildup.astype(float)),
                   cmap="autumn", alpha=0.7)
axes[1].plot(cx, cy, 'r*', markersize=15)
axes[1].set_title("Red = strandskydd zone\nYellow = new built-up"); axes[1].axis("off")

axes[2].imshow(rgb_after)
if violation_mask.any():
    axes[2].imshow(np.ma.masked_where(~violation_mask, violation_mask.astype(float)),
                   cmap="hot", alpha=0.9)
axes[2].plot(cx, cy, 'r*', markersize=15)
title_col = "firebrick" if violation_mask.any() else "gray"
axes[2].set_title(f"POTENTIAL VIOLATIONS\n{violation_mask.sum()} px  (~{violation_mask.sum()*100:.0f} m²)",
                  fontsize=11, color=title_col)
axes[2].axis("off")

handles = [mpatches.Patch(color='red',    alpha=0.5, label=f'Strandskydd ({STRANDSKYDD_DISTANCE_M} m)'),
           mpatches.Patch(color='yellow',            label='New built-up (NDBI)'),
           mpatches.Patch(color='orange',            label='Potential violation')]
axes[2].legend(handles=handles, loc='lower left', fontsize=8)

plt.suptitle(f"Strandskydd Violation Analysis — {date_before} → {date_after}", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

## 9. Interactive Map

In [ ]:
m = folium.Map(location=[POI_LAT, POI_LON], zoom_start=14, tiles=None, control_scale=True)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Satellite (Esri)", overlay=False, control=True
).add_to(m)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap", overlay=False).add_to(m)

if water_patch_wgs84 is not None:
    folium.GeoJson(
        water_patch_wgs84.__geo_interface__,
        name=f"Water bodies — Prithvi ({date_after})",
        style_function=lambda x: {"fillColor":"#3399FF","color":"#0066CC","weight":1,"fillOpacity":0.55},
        tooltip="Water body — Prithvi-EO-2.0 detection"
    ).add_to(m)

if strandskydd_patch_gdf is not None:
    folium.GeoJson(
        strandskydd_patch_gdf.__geo_interface__,
        name=f"Strandskydd zone — {STRANDSKYDD_DISTANCE_M} m (Prithvi-derived)",
        style_function=lambda x: {"fillColor":"#FF4444","color":"#CC0000","weight":1,"fillOpacity":0.2},
        tooltip=f"Protected zone — {STRANDSKYDD_DISTANCE_M} m from Prithvi water"
    ).add_to(m)

folium.Marker(
    location=[POI_LAT, POI_LON],
    popup=folium.Popup(POI_LABEL, max_width=300),
    tooltip="Point of Interest — click for details",
    icon=folium.Icon(color="red", icon="home", prefix="fa")
).add_to(m)

folium.Circle(location=[POI_LAT, POI_LON], radius=200,
              color="red", weight=1.5, fill=False,
              tooltip="200 m context radius").add_to(m)

folium.WmsTileLayer(
    url="https://nvpub.vic-metria.nu/arcgis/services/Strandskydd/MapServer/WmsServer",
    name="Strandskydd — Naturvårdsverket (official WMS)",
    layers="0", fmt="image/png", transparent=True,
    overlay=True, control=True, opacity=0.6, attr="Naturvårdsverket"
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
plugins.Fullscreen().add_to(m)

print(f"Map ready — centred on POI ({POI_LAT}°N, {POI_LON}°E)")
m

## Summary & Stage 3 Roadmap

### Stage 2 Complete

| Step | Detail |
|---|---|
| Foundation model | Prithvi-EO-2.0-300M (Sen1Floods11 fine-tune) |
| Water detection | Semantic segmentation — class 1 = water |
| Strandskydd zone | 100 m buffer from Prithvi water mask |
| Change detection | NDBI (After − Before) on POI patch |
| Violation signal | New built-up pixels ∩ strandskydd zone |

### What this demonstrates

This pipeline shows how a geospatial foundation model can automate a task  
that currently requires manual inspection: identifying potential strandskydd  
violations from satellite imagery — with a clear, auditable evidence trail  
(before/after imagery, water detection, change signal).

### Stage 3 options

- **Fine-tune Prithvi** on Lantmäteriet building footprints for pixel-precise building detection (requires GPU)  
- **Scale to full island** — Dask-distributed tiling over all of Orust or Västra Götaland  
- **Model serving** — wrap inference in a REST API on OpenShift using OpenVINO or Triton  
- **Case officer UI** — simple web interface to review flagged locations on a map